# Feature Engineering: Daily Time Series

In [13]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

In [14]:
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / 'outputs' / 'data' / 'ridership_clean.parquet'
ridership_2023_clean = pd.read_parquet(DATA_PATH)
print(f"{len(ridership_2023_clean):,} trips loaded")

30,512,679 trips loaded


### Build Daily Feature Matrix

Call the full pipeline from `src/feature_engineering`: aggregate, calendar features, lag features, rolling features. Rows with NaN from lag/rolling creation are dropped (we lose the first 28 days).

In [15]:
from src.feature_engineering import build_daily_features

df = build_daily_features(ridership_2023_clean)
print(f"{len(df)} days after dropping NaN rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
df.head()

del ridership_2023_clean

2134 days after dropping NaN rows
Date range: 2020-01-29 to 2025-12-31

Columns (26):
['trip_count', 'mean_duration_min', 'median_duration_min', 'pct_annual_member', 'pct_peak_hour', 'day_of_week', 'month', 'day_of_year', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos', 'is_weekend', 'is_holiday', 'trip_count_lag1', 'trip_count_lag7', 'trip_count_lag14', 'trip_count_lag28', 'trip_count_rmean7', 'trip_count_rstd7', 'trip_count_rmean14', 'trip_count_rstd14', 'trip_count_rmean28', 'trip_count_rstd28']


### Feature Overview

In [16]:
df.describe().round(2)

,trip_count,mean_duration_min,median_duration_min,pct_annual_member,pct_peak_hour,day_of_week,month,day_of_year,day_of_week_sin,day_of_week_cos,...,trip_count_lag1,trip_count_lag7,trip_count_lag14,trip_count_lag28,trip_count_rmean7,trip_count_rstd7,trip_count_rmean14,trip_count_rstd14,trip_count_rmean28,trip_count_rstd28
count,2134.00,2134.00,2134.00,2134.00,2134.00,2134.0,2134.00,2134.00,2134.00,2134.00,...,2134.00,2134.00,2134.00,2134.00,2134.00,2134.00,2134.00,2134.00,2134.00,2134.00
mean,14257.23,13.46,11.39,0.59,0.54,3.0,6.53,183.46,-0.00,-0.00,...,14253.19,14213.96,14147.94,14009.16,14233.19,2636.09,14203.96,2835.42,14140.31,3079.42
std,9623.79,1.82,1.90,0.29,0.07,2.0,3.40,103.89,0.71,0.71,...,9627.59,9652.42,9698.41,9788.57,9232.91,1423.50,9181.33,1290.12,9138.03,1249.42
min,155.00,9.85,8.18,0.00,0.29,0.0,1.00,1.00,-0.97,-0.90,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,6179.25,12.00,9.93,0.37,0.48,1.0,4.00,94.00,-0.78,-0.90,...,6163.75,6119.00,5911.75,5714.50,6298.82,1589.61,6286.54,1861.14,6266.89,2144.47
50%,12570.50,13.23,11.04,0.69,0.56,3.0,7.00,183.00,0.00,-0.22,...,12570.50,12419.50,12231.50,12010.00,13212.36,2398.62,13451.46,2749.73,13122.79,3191.00
75%,21349.00,14.42,12.20,0.81,0.60,5.0,9.00,272.00,0.78,0.62,...,21349.00,21349.00,21349.00,21349.00,21368.46,3594.80,21183.59,3733.17,21128.25,3896.74
max,44137.00,20.46,19.88,0.97,0.75,6.0,12.00,366.00,0.97,1.00,...,44137.00,44137.00,44137.00,44137.00,38631.43,8543.70,37495.57,7435.36,37262.68,7939.13


In [17]:
# Check for any remaining NaN
nan_counts = df.isna().sum()
if nan_counts.sum() == 0:
    print("There are no missing values. The feature matrix is clean.")
else:
    print("Missing values:")
    print(nan_counts[nan_counts > 0])

There are no missing values. The feature matrix is clean.


In [18]:
# Export to parquet (preserving Trip Id index)
output_path = PROJECT_ROOT / 'outputs' / 'data' / 'ridership.parquet'

# Use snappy compression for a balance of speed and size
# index=True keeps Trip Id in the file for traceability
df.to_parquet(output_path, compression='snappy', index=True)

del df